# Notebook for the bispectrum inversion on $\mathbb{Z}/n\mathbb{Z}$, any commutative group and the dihedral group $D_n$

In [2]:
# Imports

import numpy as np
import scipy as sc

## Inversion on $\mathbb{Z}/n\mathbb{Z}$

### Usual DFT in $\mathcal{O}(n^2)$

In [3]:
f = np.random.randn(10)


def DFT(f):
    # Computes the 1d DFT of a signal f:Z/nZ->C. Returns fhat:Z/nZ->C.
    n = len(f)
    fhat = np.zeros(n) * 1j
    for i in range(n):
        for j in range(n):
            fhat[i] += f[j] * np.exp(2 * np.pi * 1j * j * i / n)
    return fhat


fhat = DFT(f)
fhat

array([ 2.55880265+0.00000000e+00j,  3.9793941 -2.80085761e+00j,
       -1.80994021+7.56415800e-01j, -2.60804417+1.72868874e+00j,
        2.94671448-3.38639581e+00j,  1.52003701+6.55279509e-16j,
        2.94671448+3.38639581e+00j, -2.60804417-1.72868874e+00j,
       -1.80994021-7.56415800e-01j,  3.9793941 +2.80085761e+00j])

### Usefsul Bispectrum coefficients and inversion algorithm

In [4]:
def bispectrum_1d(fhat):
    """
    Compute bispectrum beta using 1d DFT.
    Input: The 1d Fourier transform on Z/nZ.
    Returns: Only the |G| bispectrum elements needed for completeness.
    These are given by beta[0,0], beta[0, 1] and beta[1, i-1] for i \in {1,2,...,n-2}
    """
    n = len(fhat)
    beta = np.zeros(n) * 1j
    beta[0] = fhat[0] * fhat[0] * np.conj(fhat[0])  # beta[0, 0]
    beta[1] = fhat[0] * fhat[1] * np.conj(fhat[1])  # beta[0, 1]
    for i in range(2, n):
        beta[i] = fhat[1] * fhat[i - 1] * np.conj(fhat[i])  # beta[1, i-1]
    return beta


def bispectrum_invert_1d(beta):
    """
    Invert the bispectrum of a 1D signal.
    Input:
    Returns the 1d original DFT up to indeterminacy.
    The method uses only |G| bispectral elements.
    Kakarala, R. Triple correlation on groups. PhD thesis, UC Irvine, 1992
    """
    n = len(beta)
    fhat = np.zeros(n) * 1j
    r = np.abs(beta[0]) ** (1 / 3)
    theta = np.angle(beta[0])
    fhat[0] = r * np.exp(1j * theta)
    fhat[1] = np.sqrt(beta[1] / fhat[0])
    for i in range(2, n):
        fhat[i] = np.conj(beta[i] / (fhat[1] * fhat[i - 1]))
    return fhat

### Sanity check function
Verifies that the output of the inversion algorithm and the initial Fourier transform are similar.

In [5]:
def find_indeterminacy_1d(fhat, fhat2):
    """
    Input: Two Fourier transforms from signals similar to group action.
    Output: The first signal fhat rotated.
    """
    n = len(fhat)
    rho_h = fhat2[1] / fhat[1]
    temp = np.zeros(n) * 1j
    for i in range(n):
        temp[i] = fhat[i] * (rho_h**i)
    return temp

## Commutative groups $G = \oplus_{l=1}^L \mathbb{Z}/n_l\mathbb{Z}$

### DFT in $\mathcal{O}(|G|^2)$

In [6]:
def DFT_com(f):
    """
    Computes the G-DFT of a signal f:G->C where G is the sum of finitely many cyclic groups.
    Input: a signal f:G->C.
    Returns fhat:G->C.
    """
    N = np.shape(f)
    L = len(N)
    fhat = np.zeros(N) * 1j
    for i in np.ndindex(N):
        for j in np.ndindex(N):
            rho = 1
            for l in range(L):
                rho *= np.exp(2 * np.pi * 1j * j[l] * i[l] / N[l])
            fhat[i] += f[j] * rho
    return fhat


def _bispectrum_nd(fhat, beta):
    """
    In-place call of bispectrum_nd(fhat).
    """
    N = np.shape(fhat)
    Nsub = tuple(list(N)[:-1])
    L = len(N)
    if L == 1:
        beta[:] = bispectrum_1d(fhat)
        return beta
    # Recursive call to solve the case of dimension L - 1
    beta[..., 0] = _bispectrum_nd(fhat[..., 0], beta[..., 0])
    el = [0] * L
    nl = N[-1]
    el[L - 1] = 1
    elt1 = tuple(el)
    beta[elt1] = fhat[tuple([0] * L)] * fhat[elt1] * np.conj(fhat[elt1])
    elt = elt1
    for i in range(1, nl):
        oldelt = elt
        el[L - 1] = i
        elt = tuple(el)
        if i > 1:
            beta[elt] = fhat[elt1] * fhat[oldelt] * np.conj(fhat[elt])
        for k in np.ndindex(Nsub):
            if sum(k) > 0:
                kp = list(k)
                k = tuple(kp + [0])
                ki = tuple(kp + [i])
                beta[ki] = fhat[k] * fhat[elt] * np.conj(fhat[ki])
    return beta


def bispectrum_nd(fhat):
    """
    Input: The Fourier transform fhat defined on a commutative group G.
    Returns:  The |G| bispectral coefficients needed to obtain a complete transformation.
    """
    N = np.shape(fhat)
    beta = np.zeros(N) * 1j
    return _bispectrum_nd(fhat, beta)

In [7]:
def _bispectrum_invert_nd(beta, fhat):
    """
    In-place version of bispectrum_invert_nd(beta, fhat)
    """
    N = np.shape(fhat)
    nl = N[-1]
    Nsub = tuple(list(N)[:-1])
    L = len(N)

    if L == 1:
        fhat[:] = bispectrum_invert_1d(beta)
        return fhat

    fhat[..., 0] = _bispectrum_invert_nd(beta[..., 0], fhat[..., 0])
    el = [0] * L
    el[L - 1] = 1
    elt1 = tuple(el)
    elt = elt1
    fhat[elt1] = np.sqrt(beta[elt1] / fhat[tuple([0] * L)])
    for i in range(1, nl):
        oldelt = elt
        el[L - 1] = i
        elt = tuple(el)
        if i > 1:
            fhat[elt] = np.conj(beta[elt] / (fhat[elt1] * fhat[oldelt]))
        for k in np.ndindex(Nsub):
            if sum(k) > 0:
                kp = list(k)
                k = tuple(kp + [0])
                ki = tuple(kp + [i])
                fhat[ki] = np.conj(beta[ki] / (fhat[k] * fhat[elt]))
    return fhat


def bispectrum_invert_nd(beta):
    """
    Input: |G| bispectral coefficients.
    Output: The Fourier transform of the original signal.
    """
    N = np.shape(beta)
    fhat = np.zeros(N) * 1j
    return _bispectrum_invert_nd(beta, fhat)


def find_indeterminacy(fhat, fhat2):
    """
    G: commutative group
    Given the true Fourier transform fhat and the Fourier transform fhat2
    obtained after the bispectrum inversion, find the indeterminacy.
    """
    N = np.shape(fhat)
    L = len(N)
    temp = np.zeros(N) * 1j
    rho_h = np.zeros(L) * 1j
    el = [0] * L
    el[0] = 1
    elt = tuple(el)
    rho_h[0] = fhat2[elt] / fhat[elt]

    for l in range(1, L):
        el[l - 1] = 0
        el[l] = 1
        elt = tuple(el)
        rho_h[l] = fhat2[elt] / fhat[elt]

    for k in np.ndindex(N):
        rho = 1
        for l in range(L):
            rho = rho * (rho_h[l] ** (k[l]))
        temp[k] = fhat[k] * rho
    return temp

In [8]:
f = np.random.randn(2, 3, 3)
fhat = DFT_com(f)
print(fhat)

beta = bispectrum_nd(fhat)
# print(beta)
fhat2 = bispectrum_invert_nd(beta)
print(fhat2)
find_indeterminacy(fhat, fhat2)

[[[ 4.4747072 +0.00000000e+00j  1.01887473+8.44393583e+00j
    1.01887473-8.44393583e+00j]
  [ 1.07525643+2.47662715e+00j  1.21872872-1.77644701e+00j
   -0.921215  -3.20896126e+00j]
  [ 1.07525643-2.47662715e+00j -0.921215  +3.20896126e+00j
    1.21872872+1.77644701e+00j]]

 [[ 1.30349357+1.94180831e-16j -0.69139379-1.58357222e+00j
   -0.69139379+1.58357222e+00j]
  [ 3.98869416+1.24287951e+00j  2.86943964-5.03242584e+00j
   -0.67272865+1.02280127e+00j]
  [ 3.98869416-1.24287951e+00j -0.67272865-1.02280127e+00j
    2.86943964+5.03242584e+00j]]]
[[[ 4.4747072 +0.j          8.50518419+0.j
   -2.99813779+7.9592291j ]
  [ 2.69997379+0.j         -1.94929477+0.91723147j
    3.11238576+1.20794218j]
  [-2.54362376+0.9054483j  -1.15088876-3.13393116j
   -0.95268492+1.93221508j]]

 [[ 1.30349357+0.j         -1.65499366+0.49671154j
    1.04822421-1.37366385j]
  [ 2.72855175-3.16376926j -5.01892919+2.89298177j
   -0.40737385-1.15444048j]
  [-3.63153101-2.06552973j  1.14680664+0.42839329j
   -2.9841

array([[[ 4.4747072 +0.00000000e+00j,  8.50518419-2.22044605e-16j,
         -2.99813779+7.95922910e+00j],
        [ 2.69997379-1.11022302e-16j, -1.94929477+9.17231474e-01j,
          3.11238576+1.20794218e+00j],
        [-2.54362376+9.05448299e-01j, -1.15088876-3.13393116e+00j,
         -0.95268492+1.93221508e+00j]],

       [[ 1.30349357+2.46519033e-32j, -1.65499366+4.96711536e-01j,
          1.04822421-1.37366385e+00j],
        [ 2.72855175-3.16376926e+00j, -5.01892919+2.89298177e+00j,
         -0.40737385-1.15444048e+00j],
        [-3.63153101-2.06552973e+00j,  1.14680664+4.28393287e-01j,
         -2.98417655+4.96524763e+00j]]])

$$
\mathcal{F}(f)_\rho = \sum_{g\in G} f(g) \cdot \rho(g)
$$
where the 2d irreps are given by
$$
\rho_k(a^lx^m)=\begin{bmatrix}
        \cos(\omega_l k)&-\sin(\omega_l k)\\
        \sin(\omega_l k)&\cos(\omega_l k)
    \end{bmatrix}
    \begin{bmatrix}
        1 & 0\\
        0 & -1\\
    \end{bmatrix}^m
$$ 
for $k = 1,2,...\lfloor \frac{n-1}{2}\rfloor$.
The 1d irreps are given by 
* $\rho_{0}(g)=1$ for all $g\in D_n$
* $\rho_{01}(g)=\begin{cases}
        1 \text{ if } g\in \langle a\rangle,\\
        -1 \text{ otherwise.}
    \end{cases}$
* If $n$ even, $\rho_{02}(g)=\begin{cases}
        1 \text{ if } g\in \langle a^2,\ x\rangle,\\
        -1 \text{ otherwise.}
    \end{cases}$
* If $n$ even, $\rho_{03}(g)=\begin{cases}
        1 \text{ if } g\in \langle a^2,\ ax\rangle,\\
        -1 \text{ otherwise.}
    \end{cases}$

In [9]:
def DFT_dihedral(f):
    """
    Input: A function f:G->C where G is the dihedral group D_n (symmetries of the n-gon).
    G = {e, a, a^2...,a^{n-1},x, ax, a^2x,...,a^{n-1}x}.
    Output: returns the Fourier transform.
    """
    n = int(len(f) / 2)
    n2d = int(np.floor((n - 1) / 2))
    # fhat1d = np.zeros(n1d)
    fhat = np.zeros((2, 2, n2d + 1))
    # the coeffs for the 1d irreps are stored in fhat[..., 0]
    # the coeffs for the 2d irreps are stored in fhat[..., 1:n2d+1 (included)]
    for j in range(n):
        fhat[0, 0, 0] += f[j] + f[j + n]
        fhat[1, 0, 0] += f[j] - f[j + n]
    if n % 2 == 0:
        for j in range(0, n, 2):
            fhat[0, 1, 0] += f[j] - f[j + 1] + f[j + n] - f[j + 1 + n]
            fhat[1, 1, 0] += f[j] - f[j + 1] - f[j + n] + f[j + 1 + n]

    for i in range(1, n2d + 1):
        for j in range(n):
            omega = 2 * np.pi * i * j / n
            rho = np.array(
                [[np.cos(omega), -np.sin(omega)], [np.sin(omega), np.cos(omega)]]
            )
            fhat[..., i] += f[j] * rho
            rho[:, 1] *= -1
            fhat[..., i] += f[j + n] * rho
    return fhat


n = 8
f = np.random.randn(2 * n)
fhat = DFT_dihedral(f)
fhat

array([[[-0.36072721, -1.15806986, -0.49860132,  1.48770366],
        [-5.32728905,  1.40180527,  6.30579994, -4.12424398]],

       [[ 1.71076842,  1.35839595, -2.39384576, -2.45535177],
        [ 0.29928345,  1.04763531, -2.5545035 ,  4.72114947]]])

In [10]:
print(np.linalg.eigvals(fhat[...,1]))
print(fhat[...,1])
fhat[...,1]@fhat[...,1].T

[-1.82170827  1.71127372]
[[-1.15806986  1.40180527]
 [ 1.35839595  1.04763531]]


array([[ 3.30618382, -0.1045367 ],
       [-0.1045367 ,  2.9427793 ]])

The next cells allow to build the bispectrum on the dihedral group. The dihedral group is not commutative. The formula for the bispectrum $\beta(\Theta)$ is given by:
$$
\beta(\Theta)_{\rho_1,\rho_2} =
        \left[\mathcal{F}(\Theta)_{\rho_1}\otimes\mathcal{F}(\Theta)_{\rho_2}\right]C_{\rho_1,\rho_2}\left[\bigoplus_{\rho\in\rho_1\otimes\rho_2}\mathcal{F}(\Theta)_\rho^\dagger\right]C_{\rho_1,\rho_2}^{\dagger}
$$

In [11]:
def givens(omega):
    """
    Input: a scalar value omega.
    Return: the Givens rotation matrix associated to a rotation of amplitude omega.
    """
    return np.array([[np.cos(omega), -np.sin(omega)], [np.sin(omega), np.cos(omega)]])


def first_last_cb(n, end=False):
    """
    Input: n, for the symmetries of the n-gon.
    Return: the Clebsch-Gordan matrix of rho_0\otimes \rho_1 if end = False, rho_1\otimes \rho_{n/2-1} otherwise.
    """
    epst = 100 * np.finfo(np.float64).eps
    A = givens(2 * np.pi / n)
    B = givens(2 * np.pi / n)
    S = sc.linalg.schur(np.kron(A, B))
    CBt = S[1]
    l = []
    for i in range(4):
        if np.abs(np.abs(S[0][i, i]) - 1) < epst:
            l.append(i)
    return CBt, np.array(l, dtype=int)


def clebsch_gordan(label_1, label_2, n):
    """
    Input: n, for the symmetries of the n-gon.
    Return: the Clebsch-Gordan matrix of rho_label_1\otimes \rho_label_2 + labels of the representations in rho_label_1\otimes \rho_label_2.
    """
    indices = np.array([0, 0])
    A = givens(2 * np.pi * label_1 / n)
    B = givens(2 * np.pi * label_2 / n)
    S = sc.linalg.schur(np.kron(A, B))
    indices[0] = int(np.arccos(S[0][0, 0]) * n / (2 * np.pi)) #These are the irreps generated by label_1\otimes label_2
    indices[1] = int(np.arccos(S[0][2, 2]) * n / (2 * np.pi))
    return S[1], indices


def buildFplus(l, fhat, n, end=False):
    t = 2
    Fplus = np.zeros((4, 4))
    if end == True:
        t = int(np.floor((n - 1) / 2)) - 1
        Fplus[l[0], l[0]] = fhat[0, 1, 0]
        Fplus[l[1], l[1]] = fhat[1, 1, 0]
    else:
        Fplus[l[0], l[0]] = fhat[0, 0, 0]
        Fplus[l[1], l[1]] = fhat[1, 0, 0]

    if l[1] - l[0] == 1:
        if l[0] == 0:
            Fplus[3:, 3:] = fhat[..., t]
        else:
            Fplus[1:3, 1:3] = fhat[..., t]
    else:
        Fplus[1:3, 1:3] = fhat[..., t]
    return Fplus


def bispectrum_dihedral(fhat, n):
    """
    Input: Fourier transform over D_n
    Output:  the bispectral elements needed for completeness
    """
    #computes beta_\rho_0,\rho_0
    beta0 = fhat[0, 0, 0] ** 3
    #computes beta_\rho_1,\rho_1
    print(fhat[...,1])
    beta10 = fhat[0, 0, 0] * (fhat[..., 1] @ fhat[..., 1].T)
    n2 = int(np.floor((n - 1) / 2))
    n3 = n2
    if n % 2 > 0:
        n3 = n2 - 1
    beta1i = np.zeros((4, 4, n3))
    indices = np.zeros((2, n3), dtype = int)
    CBmatrices = np.zeros((4, 4, n3))
        
    CBmatrices[..., 0], indices[:, 0] = first_last_cb(n, end=False)
    Fplus = buildFplus(indices[:, 0], fhat, n, end=False)
    #print(Fplus, indices)
    beta1i[..., 0] = np.kron(fhat[..., 1], fhat[..., 1]) @ CBmatrices[..., 0] @ Fplus.T @ (CBmatrices[..., 0].T)
    Fplus = np.zeros((4, 4))
    for i in range(2, n2):
        CBmatrices[..., i - 1], indices[:, i - 1] = clebsch_gordan(1, i, n)
        Fplus[:2, :2] = fhat[..., indices[0, i - 1]]
        Fplus[2:, 2:] = fhat[..., indices[1, i - 1]]
        beta1i[..., i - 1] = np.kron(fhat[..., 1], fhat[..., i]) @ CBmatrices[..., i - 1] @ Fplus.T @ (CBmatrices[..., i - 1].T)
    Fplus = np.zeros((4, 4))
    if n % 2 == 0:
        CBmatrices[..., n2 - 1], indices[:, n2 - 1] = first_last_cb(n, end=True)
        Fplus = buildFplus(indices[:, n2 - 1], fhat, n, end=True)
        beta1i[..., n2 - 1] = np.kron(fhat[..., 1], fhat[..., n2]) @ CBmatrices[..., n2 - 1] @ Fplus.T @ (CBmatrices[..., n2 - 1].T)
    return (beta0, beta10, beta1i), CBmatrices, indices

beta, CBmatrices, indices = bispectrum_dihedral(fhat, n)
beta

[[-1.15806986  1.40180527]
 [ 1.35839595  1.04763531]]


(-0.0469393102188722,
 array([[-1.19263046,  0.03770923],
        [ 0.03770923, -1.06154056]]),
 array([[[-8.24010008e+00, -1.30749721e+01,  3.13718543e+01],
         [ 3.39188358e-01, -5.15074258e+01, -4.86502799e+00],
         [ 3.39188358e-01, -6.01181107e+00, -5.67030751e+00],
         [ 7.04746962e+00, -2.74585404e+01,  8.60546378e+00]],
 
        [[ 7.17320118e+00,  1.02585297e+01, -4.25858780e+01],
         [ 2.53381547e+00,  7.16199636e+00,  4.05342859e+00],
         [ 7.86703590e+00, -7.12873877e+00,  4.65962597e+00],
         [-7.13549195e+00,  1.31861821e+01, -7.81880592e+00]],
 
        [[ 7.17320118e+00,  7.21464780e+00, -1.57897922e+01],
         [ 7.86703590e+00,  2.32242536e+01, -1.24728056e+01],
         [ 2.53381547e+00,  2.79351586e+01, -1.03296586e+01],
         [-7.13549195e+00,  1.32792254e+01,  2.80415609e+01]],
 
        [[ 5.82041569e+00,  7.73244300e+00,  2.04574241e+01],
         [-6.30765633e-01, -1.05953969e+01,  1.59241358e+01],
         [-6.30765633e-01, 

In [12]:
def extract_fhat_from_Fplus(fhat, Fplus, indices, i, n2):
    if i == 2:
        if indices[1]-indices[0] == 1:
            if indices[0] == 0:
                fhat[..., i] = Fplus[2:, 2:]
            else:
                fhat[..., i] = Fplus[:2, :2]
        else:
            fhat[..., i] = Fplus[1:3, 1:3]
        fhat[1, 0, 0] = Fplus[indices[1], indices[1]]
    elif i == n2 + 1:
        fhat[0, 1, 0] = Fplus[indices[0], indices[0]]
        fhat[1, 1, 0] = Fplus[indices[1], indices[1]]
    else:
        if indices[0] > indices[1]:
            fhat[..., i] = Fplus[:2, :2]
        else:
            fhat[..., i] = Fplus[2:, 2:]
    return
def bispectrum_invert_dihedral(beta, n, CBmatrices, indices, f):
    
    beta0 = beta[0]
    beta10 = beta[1]
    beta1i = beta[2]
    n2 = int(np.floor((n - 1) / 2))
    fhat = np.zeros((2, 2, n2 + 1))
    fhat[0, 0, 0] = (abs(beta0))**(1/3) * np.sign(beta0)
    values, vectors = np.linalg.eigh(beta10 / fhat[0, 0, 0])
    fhat[..., 1] =  f[...,1] #@ givens(2*np.pi/n)#vectors @ np.diag(np.sqrt(values)) @ vectors.T
    #print(np.linalg.eigvals(fhat[...,1]))
    for i in range(2, n2 + 1):
        Fplus = (CBmatrices[...,i - 2].T @ np.linalg.inv(np.kron(fhat[..., 1], fhat[..., i - 1])) @ beta1i[..., i - 2] @ CBmatrices[...,i - 2]).T
        extract_fhat_from_Fplus(fhat, Fplus, indices[:, i - 2], i, n2)
    if n % 2 == 0:
        Fplus = (CBmatrices[...,n2 - 1].T @ np.linalg.inv(np.kron(fhat[..., 1], fhat[..., n2])) @ beta1i[..., n2 - 1] @ CBmatrices[...,n2 - 1]).T
        extract_fhat_from_Fplus(fhat, Fplus, indices[:, n2 - 1], n2 + 1, n2)
    return fhat

In [13]:
def find_indeterminacy_dihedral(fhat, fhat2, n):
    n2 = int(np.floor((n - 1) / 2))
    temp = np.zeros((2, 2, n2 + 1))
    temp[0, 0, 0] = fhat[0, 0, 0]
    rho_h =  fhat2[..., 1] @ np.linalg.inv(fhat[...,1])
    print(rho_h)
    l = 0
    if np.linalg.det(rho_h) < 0:
        l = 1
    omega = np.arccos(rho_h[0, 0])
    print(omega / np.pi/2 * n)
    for i in range(1, n2+1):
        M = np.array([[np.cos(omega * i), -np.sin(omega * i)],[np.sin(omega * i), np.cos(omega * i)]])
        if l == 1:
            M[:, 1] *= -1
        temp[..., i] = M @ fhat[..., i] 
    return temp

In [14]:
fhat2 = bispectrum_invert_dihedral(beta, n, CBmatrices, indices, fhat)
temp = find_indeterminacy_dihedral(fhat, fhat2, n)
temp

[[1. 0.]
 [0. 1.]]
0.0


array([[[-0.36072721, -1.15806986, -0.49860132,  1.48770366],
        [ 0.        ,  1.40180527,  6.30579994, -4.12424398]],

       [[ 0.        ,  1.35839595, -2.39384576, -2.45535177],
        [ 0.        ,  1.04763531, -2.5545035 ,  4.72114947]]])

In [15]:
print(fhat2)
print(fhat)

[[[-0.36072721 -1.15806986 -0.49860132  1.48770366]
  [-5.32728905  1.40180527  6.30579994 -4.12424398]]

 [[ 1.71076842  1.35839595 -2.39384576 -2.45535177]
  [ 0.29928345  1.04763531 -2.5545035   4.72114947]]]
[[[-0.36072721 -1.15806986 -0.49860132  1.48770366]
  [-5.32728905  1.40180527  6.30579994 -4.12424398]]

 [[ 1.71076842  1.35839595 -2.39384576 -2.45535177]
  [ 0.29928345  1.04763531 -2.5545035   4.72114947]]]


In [16]:
fhat

array([[[-0.36072721, -1.15806986, -0.49860132,  1.48770366],
        [-5.32728905,  1.40180527,  6.30579994, -4.12424398]],

       [[ 1.71076842,  1.35839595, -2.39384576, -2.45535177],
        [ 0.29928345,  1.04763531, -2.5545035 ,  4.72114947]]])

In [17]:
r = (1, 2)
k = r
r = (2, 3)
np.arange(0, 10, 2)

array([0, 2, 4, 6, 8])

In [18]:
f = np.random.randn(3, 3, 3)
"""
print(np.shape(f))
f[tuple([0]*2)]
2*(0, 1, 0)

print([i for i in np.ndindex((3,3,3))])
"""


def cc(A):
    print(np.shape(A))
    A[:] = 2 * A
    return A


A = np.random.randn(3, 3)
print(A)
cc(A[:, 0])
print(A)
f[(0, 0)]
k = tuple([1] * 3)
k2 = [2] * 3
k
l = list((1, 2))
# l.append(2)
# print(l+[2])
print(tuple(l + [2]))
print(tuple(l + [3]))
print(sum(k))
k

[[-1.11580767 -0.48850027 -0.45042231]
 [-0.72946104 -0.2083292   0.07357976]
 [ 0.21242899 -1.00873835  0.78519843]]
(3,)
[[-2.23161533 -0.48850027 -0.45042231]
 [-1.45892208 -0.2083292   0.07357976]
 [ 0.42485799 -1.00873835  0.78519843]]
(1, 2, 2)
(1, 2, 3)
3


(1, 1, 1)

In [19]:
A = np.random.randn(2, 3, 2)
print(np.shape(A))
print(A)
print(A[..., 0])
print(A[0, 0, 1])

(2, 3, 2)
[[[ 1.71272313 -0.61746042]
  [-0.65557549  0.0128457 ]
  [ 0.49441772 -1.65687577]]

 [[ 1.69704794 -0.96660923]
  [ 0.14696536  0.57140467]
  [-0.43759528 -0.47482731]]]
[[ 1.71272313 -0.65557549  0.49441772]
 [ 1.69704794  0.14696536 -0.43759528]]
-0.6174604185291953


In [20]:
n = 20
g = 1
omega = 2 * np.pi * 9 / n * g
omega2 = 2 * np.pi * 9 / n * g
print(omega)
A1 = np.array([[np.cos(omega), -np.sin(omega)], [np.sin(omega), np.cos(omega)]])
A2 = np.array([[np.cos(omega2), -np.sin(omega2)], [np.sin(omega2), np.cos(omega2)]])
B = np.round(sc.linalg.schur(np.kron(A1, A2))[0], decimals=3)
print(B)
print(np.arccos(B[0, 0]) * n / (2 * np.pi) / g)
print(np.arccos(B[2, 2]) * n / (2 * np.pi) / g)
print(np.round(sc.linalg.schur(np.kron(A1, A2))[1], decimals=3))
np.array([1,2], dtype=int)

2.827433388230814
[[ 1.    -0.    -0.    -0.   ]
 [ 0.     0.809  0.588 -0.   ]
 [ 0.    -0.588  0.809 -0.   ]
 [ 0.     0.     0.     1.   ]]
0.0
2.0000920296980045
[[-0.707 -0.705 -0.053  0.   ]
 [ 0.     0.053 -0.705  0.707]
 [-0.     0.053 -0.705 -0.707]
 [-0.707  0.705  0.053  0.   ]]


array([1, 2])

In [47]:
np.ravel(np.ones((3,3,3)))
import torch
A = torch.ones(2,2)
for i in np.ndindex(A.shape):
    print(i)
v = torch.ones(4)
A.shape[1]
a = torch.ones(1)*3
torch.cat((a, v, torch.ravel(A)))

(0, 0)
(0, 1)
(1, 0)
(1, 1)


tensor([3., 1., 1., 1., 1., 1., 1., 1., 1.])

In [41]:
import torch
import numpy as np
import scipy as sc

In [44]:
def fourier_transform_dihedral(f):
        """
        Input: A function f:G->C where G is the dihedral group D_n (symmetries of the n-gon).
        G = {e, a, a^2...,a^{n-1}, x, ax, a^2x,...,a^{n-1}x}.
        Output: returns the Fourier transform.
        """
        n = int(len(f)/ 2)
        n2d = int(np.floor((n - 1) / 2))
        fhat = torch.zeros(2, 2, n2d + 1)
        # the coeffs for the 1d irreps are stored in fhat[..., 0]
        # the coeffs for the 2d irreps are stored in fhat[..., 1:n2d+1 (included)]
        for j in range(n):
            fhat[0, 0, 0] += f[j] + f[j + n]
            fhat[1, 0, 0] += f[j] - f[j + n]
        if n % 2 == 0:
            for j in range(0, n, 2):
                fhat[0, 1, 0] += f[j] - f[j + 1] + f[j + n] - f[j + 1 + n]
                fhat[1, 1, 0] += f[j] - f[j + 1] - f[j + n] + f[j + 1 + n]
        
        for i in range(1, n2d + 1):
            for j in range(n):
                omega = 2 * np.pi * i * j / n
                rho = torch.tensor(
                    [[np.cos(omega), -np.sin(omega)], [np.sin(omega), np.cos(omega)]]
                )
                rho1 = rho.clone()
                fhat[..., i] += f[j] * rho
                rho1[:, 1]    *= -1
                fhat[..., i] += f[j + n] * rho1
        
        return fhat
        
f = torch.randn(16, requires_grad=True)
#fhat = fourier_transform_dihedral(f)

In [32]:
print(f.grad)

tensor([ 6.,  0.,  2.,  0.,  0.,  2.,  0., -2.])


In [31]:
fourier_transform_dihedral(f).sum().backward()

In [38]:
def givens(omega):
    return np.array([[np.cos(omega), -np.sin(omega)], [np.sin(omega), np.cos(omega)]])


def first_last_cb(n, end=False):
    epst = 100 * np.finfo(np.float64).eps
    A = givens(2 * np.pi / n)
    B = givens(2 * np.pi / n)
    S = sc.linalg.schur(np.kron(A, B))
    CBt = torch.tensor(S[1])
    l = []
    for i in range(4):
        if np.abs(np.abs(S[0][i, i]) - 1) < epst:
            l.append(i)
    return CBt, np.array(l, dtype=int)


def clebsch_gordan(label_1, label_2, n):
    indices = np.array([0, 0])
    A = givens(2 * np.pi * label_1 / n)
    B = givens(2 * np.pi * label_2 / n)
    S = sc.linalg.schur(np.kron(A, B))
    indices[0] = int(np.arccos(S[0][0, 0]) * n / (2 * np.pi)) #These are the irreps generated by label_1\otimes label_2
    indices[1] = int(np.arccos(S[0][2, 2]) * n / (2 * np.pi))
    return torch.tensor(S[1]), indices


def buildFplus(l, fhat, n, end=False):
    t = 2
    Fplus = torch.zeros(4, 4)
    if end == True:
        t = int(np.floor((n - 1) / 2)) - 1
        Fplus[l[0], l[0]] = fhat[0, 1, 0]
        Fplus[l[1], l[1]] = fhat[1, 1, 0]
    else:
        Fplus[l[0], l[0]] = fhat[0, 0, 0]
        Fplus[l[1], l[1]] = fhat[1, 0, 0]

    if l[1] - l[0] == 1:
        if l[0] == 0:
            Fplus[3:, 3:] = fhat[..., t]
        else:
            Fplus[1:3, 1:3] = fhat[..., t]
    else:
        Fplus[1:3, 1:3] = fhat[..., t]
    return Fplus

In [100]:


def bispectrum_dihedral( x, n):
        """
        Input: Fourier transform over D_n
        Output:  the bispectral elements needed for completeness
        """
        fhat = fourier_transform_dihedral(x)
        #computes beta_\rho_0,\rho_0
        beta0 = torch.ones(1) * fhat[0, 0, 0] ** 3
        #computes beta_\rho_1,\rho_1
        beta10 = fhat[0, 0, 0] * (fhat[..., 1] @ fhat[..., 1].T)
        n2 = int(np.floor((n - 1) / 2))
        n3 = n2
        if n % 2 > 0:
            n3 = n2 - 1
        beta1i = torch.zeros(4, 4, n3)
        indices = np.zeros(2, dtype = int)
        CBmatrices = torch.zeros(4, 4)
            
        CBmatrices, indices = first_last_cb(n, end=False)
        Fplus = buildFplus(indices, fhat, n, end=False)
        beta1i[..., 0] = torch.kron(fhat[..., 1], fhat[..., 1]) @ CBmatrices @ Fplus.T @ (CBmatrices.T)
        Fplus = torch.zeros(4, 4)
        CBmatrices = CBmatrices.clone()
        for i in range(2, n2):
            a, b = clebsch_gordan(1, i, n)
            CBmatrices = a
            indices = b
            Fplus[:2, :2] = fhat[..., indices[0]]
            Fplus[2:, 2:] = fhat[..., indices[1]]
            beta1i[..., i - 1] = torch.kron(fhat[..., 1], fhat[..., i]) @ CBmatrices @ Fplus.T @ (CBmatrices.T)
        
        Fplus = torch.zeros(4, 4)
        CBmatrices = CBmatrices.clone()
        if n % 2 == 0:
            CBmatrices, indices= first_last_cb(n, end=True)
            Fplus = buildFplus(indices, fhat, n, end=True)
            beta1i[..., n2 - 1] = torch.kron(fhat[..., 1], fhat[..., n2]) @ CBmatrices @ Fplus.T @ (CBmatrices.T)
        
        return torch.cat((beta0,torch.ravel(beta10),torch.ravel(beta1i)))

In [101]:
beta = bispectrum_dihedral(f, 8)

In [168]:
f = torch.ones(2,2,2)*3
f2 = torch.ones(2,2,2)*2
print(f[:, :, 0].unsqueeze(2) * f * f2)

tensor([[[18., 18.],
         [18., 18.]],

        [[18., 18.],
         [18., 18.]]])


In [161]:
r = torch.ones(2,2,2,2,1)
f = torch.randn(2,2,2)
r.sum()
#f = torch.randn(8)
f.sum(axis = 2)
a = torch.arange(3)
A = torch.randn(2,2)
B = torch.randn(2,2)

m, n = A.shape
p, q = B.shape

# Reshape A and B to make them compatible with torch.kron
A_reshaped = A.view(m, 1, n, 1)
B_reshaped = B.view(1, p, 1, q)

# Perform vectorized Kronecker product using torch.einsum
C = torch.einsum('ij,kl->ikjl', A,B).view(4,4)
C = (A[:, None, :, None] * B[None, :, None, :]).reshape((4, 4))
print(C)
print(torch.kron(A, B))
A = torch.randn(2,2).unsqueeze(2)
B = torch.randn(2,2, 3)
C = torch.randn(2,2, 3)
D = torch.cat((A, B, C), dim =2)


tensor([[ 0.2051,  0.3591,  0.2425,  0.4246],
        [ 0.0523, -2.7625,  0.0619, -3.2661],
        [ 0.0301,  0.0526, -0.0793, -0.1389],
        [ 0.0077, -0.4050, -0.0202,  1.0683]])
tensor([[ 0.2051,  0.3591,  0.2425,  0.4246],
        [ 0.0523, -2.7625,  0.0619, -3.2661],
        [ 0.0301,  0.0526, -0.0793, -0.1389],
        [ 0.0077, -0.4050, -0.0202,  1.0683]])


In [142]:
f = torch.randn(4,4,16)
n = int(f.shape[2]/ 2)
n2d = int(np.floor((n - 1) / 2))
fhat = torch.zeros(f.shape[0], f.shape[1], 2, 2, n2d + 1)
# the coeffs for the 1d irreps are stored in fhat[..., 0]
# the coeffs for the 2d irreps are stored in fhat[..., 1:n2d+1 (included)]
fhat[:, :, 0, 0, 0] = f.sum(axis= 2)
fhat[:, :, 1, 0, 0] = f[:, :, :n].sum(axis= 2) - f[:, :, n:].sum(axis= 2)
if n % 2 == 0:
    fhat[:, :, 0, 1, 0] = f[:, :, 0:2*n:2].sum(axis= 2) - f[:, :, 1:2*n:2].sum(axis= 2)
    fhat[:, :, 1, 0, 0] = f[:, :, 0:n:2].sum(axis= 2) - f[:, :, 1:n:2].sum(axis= 2) - f[:, :, n:2*n:2].sum(axis= 2) + f[:, :, n+1:2*n:2].sum(axis = 2)
i_range = np.arange(1, n2d + 1)
j_range = np.arange(n)

# Create omega tensor
omega = 2 * np.pi * i_range[:, None] * j_range / n
# Create rho tensor
rho = torch.tensor(
    [
        [np.cos(omega), -np.sin(omega)],
        [np.sin(omega), np.cos(omega)]
    ]
) #rho.shape = [2, 2, n2d, n]
print(rho.shape)
rho1 = rho.clone()
rho1[:, 1] *= -1

#
fhat[..., 1:n2d+1] = torch.sum(f[:, :, None, None, None, j_range] * rho[None, None, :, :, :], dim = 5)
fhat[..., 1:n2d+1] += torch.sum(f[:, :, None, None, None, j_range + n] * rho1[None, None, :, :, :], dim = 5)
fhat


torch.Size([2, 2, 3, 8])


tensor([[[[[ 2.3280e+00, -3.1493e-01,  2.6484e+00,  3.8817e+00],
           [ 6.4508e+00,  1.8121e-01,  1.5449e+00, -5.2587e+00]],

          [[ 1.7777e+00,  1.4019e-01, -2.7467e+00,  2.7717e+00],
           [ 0.0000e+00, -2.9035e+00,  8.4158e-01, -1.6713e+00]]],


         [[[-1.1032e+00, -1.4228e+00,  1.1776e+00,  1.6480e+00],
           [ 1.0646e-01,  1.1110e+00, -3.6211e+00, -1.6094e+00]],

          [[-1.3700e+00, -7.0678e-01, -2.3740e+00,  2.1808e+00],
           [ 0.0000e+00,  5.0122e+00, -7.7476e-01, -1.8661e+00]]],


         [[[ 1.7669e+00,  9.3197e-01,  2.5593e+00,  2.2425e+00],
           [ 1.0060e+00,  2.2322e-01,  2.9972e+00, -2.0649e+00]],

          [[-5.9744e+00, -3.2404e+00, -3.4367e-01, -2.9945e-01],
           [ 0.0000e+00,  2.1423e+00, -4.6974e+00,  5.5930e-01]]],


         [[[-3.6087e+00, -6.5813e+00,  5.8984e-02, -2.2050e-01],
           [-4.1481e+00,  2.0779e+00, -2.3482e+00,  1.5000e+00]],

          [[ 4.1017e+00,  2.1223e+00,  2.4240e+00, -4.5909e+00],
     